# Conv2d sur un spectrogramme mel et autoencodeur

Suivre les formes de la représentation jusqu’à sa reconstruction.

Nous exécutons ce notebook avec un noyau Python neuf, sur CPU.

Version vérifiée : Python 3.11.9, PyTorch 2.10.0 et TorchAudio 2.10.0.

Les cellules n’installent aucun paquet et ne téléchargent aucune donnée.

Les nombres du cours viennent de cette version ; une autre version peut modifier certains arrondis ou messages.

Dans Colab, choisissons un environnement sans accélérateur, puis Exécution → Tout exécuter.

Les trous des exercices restent dans du Markdown : ils ne cassent donc pas l’exécution complète.

Les deux ondes de ce programme sont synthétiques. Elles ne sont pas les enregistrements réels du laboratoire audio.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#conv-mel-autoencodeur).

Le programme suivant est copié à l’identique depuis le cours.

In [1]:
import os
import sys
import tempfile

# À lancer dans un noyau neuf : ce tutoriel reste sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA visible :", torch.cuda.is_available())
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-pytorch-")
os.chdir(_quiz_temporary_directory.name)
import torchaudio
print("TorchAudio :", torchaudio.__version__)
assert not torch.cuda.is_available(), 'Choisir un noyau CPU neuf pour ce notebook.'

Python : 3.11.9
PyTorch : 2.10.0+cpu
CUDA visible : False
TorchAudio : 2.10.0+cpu


## Nous lisons le programme

**1. Produire des entrées contrôlées**

Deux sinusoïdes synthétiques servent au contrôle. torch.stft produit des coefficients complexes ; MelSpectrogram regroupe leur puissance avec les conventions explicitement choisies.

**2. Donner les axes au réseau**

Nous conservons 96 trames pour cette architecture. L’axe de canal est ajouté ; il ne crée pas un enregistrement stéréo.

**3. Vérifier la reconstruction**

Les convolutions transposées réagrandissent les axes. La MSE compare la carte produite à son entrée, puis un pas ajuste les poids.

In [2]:
import torch
import torchaudio
from torch import nn

torch.manual_seed(17)
torch.set_num_threads(1)
sr = 16000
t = torch.arange(sr) / sr
ondes = torch.stack([0.2 * torch.sin(2 * torch.pi * 220 * t),
                     0.2 * torch.sin(2 * torch.pi * 440 * t)])
fenetre = torch.hann_window(400)
D = torch.stft(ondes, n_fft=400, hop_length=160, window=fenetre,
               center=False, return_complex=True)
transformation = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr, n_fft=400, win_length=400, hop_length=160,
    n_mels=64, f_min=0., f_max=8000., center=False,
    mel_scale="slaney", norm="slaney", power=2.)
mel = transformation(ondes) / fenetre.sum().square()
entree = (10 * mel.clamp_min(1e-10).log10() / 100)[:, None, :, :96]
encodeur = nn.Sequential(nn.Conv2d(1, 4, 3, stride=2, padding=1),
                        nn.ReLU(), nn.Conv2d(4, 8, 3, stride=2, padding=1), nn.ReLU())
decodeur = nn.Sequential(nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1),
                        nn.ReLU(), nn.ConvTranspose2d(4, 1, 4, stride=2, padding=1))
modele = nn.Sequential(encodeur, decodeur)
print("STFT :", tuple(D.shape), "| mel :", tuple(mel.shape))
formes = []
z = entree
for nom, module in [("entrée", nn.Identity()),
                    ("conv1", encodeur[0]), ("relu1", encodeur[1]),
                    ("conv2", encodeur[2]), ("relu2", encodeur[3]),
                    ("transposée1", decodeur[0]), ("relu3", decodeur[1]),
                    ("transposée2", decodeur[2])]:
    z = module(z)
    formes.append([nom, list(z.shape)])
    print(nom, tuple(z.shape))
opt = torch.optim.Adam(modele.parameters(), lr=0.01)
opt.zero_grad(set_to_none=True)
reconstruction = modele(entree)
perte = nn.functional.mse_loss(reconstruction, entree)
avant = perte.item()
perte.backward()
opt.step()
with torch.no_grad():
    apres = nn.functional.mse_loss(modele(entree), entree).item()
print(f"MSE avant={avant:.6f}, après un pas={apres:.6f}")

STFT : (2, 201, 98) | mel : (2, 64, 98)
entrée (2, 1, 64, 96)
conv1 (2, 4, 32, 48)
relu1 (2, 4, 32, 48)
conv2 (2, 8, 16, 24)
relu2 (2, 8, 16, 24)
transposée1 (2, 4, 32, 48)
relu3 (2, 4, 32, 48)
transposée2 (2, 1, 64, 96)


MSE avant=0.875999, après un pas=0.796794


## Interpréter avant de modifier

Exécution CPU vérifiée ; les sorties sont conservées dans scripts/verify/pytorch-additions-results.json.

Les paramètres de convolution déterminent la forme, sans expliquer seuls son sens.

Une convolution transposée peut agrandir sans inverser exactement l’encodeur.

La MSE sur mel ne constitue pas une mesure complète de fidélité audio.

## Exercice 1 — Prévoir la forme d’une convolution

Nous réduisons les deux dimensions d’une carte mel. Complétons les paramètres puis vérifions la forme.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
entree = torch.zeros(2, 1, 64, 96)
conv = torch.nn.Conv2d(1, 4, kernel_size=3, stride=..., padding=...)
sortie = conv(entree)
assert tuple(sortie.shape) == (...)
```

**Piste :** Le nombre d’exemples reste deux. Les quatre filtres deviennent quatre canaux de sortie.

## Exercice 2 — Lire une perte de reconstruction

Une seule case présente une erreur de un, parmi quatre cases. Complétons le calcul de MSE.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
cible = torch.tensor([0., 0., 0., 1.])
reconstruction = torch.zeros(4)
perte = ((reconstruction - cible) ** ...).mean()
assert perte.item() == ...
```

**Piste :** La moyenne porte sur quatre erreurs au carré. Une faible MSE ne garantit pas seule une bonne écoute.

In [3]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()
print('Programme terminé ; fichiers temporaires supprimés.')

Programme terminé ; fichiers temporaires supprimés.


## Solutions — ouvrons-les après notre recherche

Les solutions utilisent leurs propres variables. Nous pouvons les copier dans de nouvelles cellules.

<details>
<summary>Afficher la solution 1 — Prévoir la forme d’une convolution</summary>

<pre><code>import torch
entree_exo = torch.zeros(2, 1, 64, 96)
conv_exo = torch.nn.Conv2d(1, 4, kernel_size=3, stride=2, padding=1)
sortie_exo = conv_exo(entree_exo)
assert tuple(sortie_exo.shape) == (2, 4, 32, 48)
print(&#x27;forme après convolution :&#x27;, tuple(sortie_exo.shape))</code></pre>

</details>

<details>
<summary>Afficher la solution 2 — Lire une perte de reconstruction</summary>

<pre><code>import torch
cible_exo = torch.tensor([0., 0., 0., 1.])
reconstruction_exo = torch.zeros(4)
perte_exo = ((reconstruction_exo - cible_exo) ** 2).mean()
assert perte_exo.item() == 0.25
print(&#x27;MSE :&#x27;, perte_exo.item())</code></pre>

</details>

